In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('SDM.db')

data = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
data

,name
0,Accessoire_Verkoop_Leverancier
1,Accessoire_Verkoop_Filiaal
2,Accessoire_Verkoop_Klant
3,Accessoire_Verkoop_Accessoire
4,Accessoire_Verkoop_Monteur
5,Accessoire_Verkoop
6,Fietsverkoop_Klant
7,Fietsverkoop_Filiaal
8,Fietsverkoop_Fiets
9,Fiets_Verkoop


In [2]:
import sqlite3
import pandas as pd
from loguru import logger

def overzetten_data():
    # Verbinding met je doel (SDM)
    conn_sdm = sqlite3.connect('SDM.db')
    
    # Configuratie: Welke bron-tabel gaat naar welke SDM-tabel?
    # Dit is waar je de 'associaties' legt door data centraal te verzamelen
    mapping = {
        'BikeToDrive_1_Accessoireverkoop.db': {
            'Klant': 'Accessoire_Verkoop_Klant',
            'Accessoire_Verkoop': 'Accessoire_Verkoop'
        },
        'BikeToDrive_2_Fietsverkoop.db': {
            'Klant': 'Fietsverkoop_Klant',
            'Fiets_Verkoop': 'Fiets_Verkoop'
        }
        # Voeg hier de rest van je 5 databases en tabellen toe op dezelfde manier
    }

    logger.info("Start overzetten data naar SDM...")

    for bron_db, tabellen in mapping.items():
        try:
            conn_bron = sqlite3.connect(bron_db)
            for bron_tabel, sdm_tabel in tabellen.items():
                # Extract
                df = pd.read_sql_query(f"SELECT * FROM {bron_tabel}", conn_bron)
                
                # Load (we gebruiken 'append' omdat de tabellen al bestaan via je CREATE-scripts)
                df.to_sql(sdm_tabel, conn_sdm, if_exists='append', index=False)
                
                logger.success(f"Overgezet: {len(df)} rijen van {bron_db}[{bron_tabel}] naar SDM[{sdm_tabel}]")
            conn_bron.close()
        except Exception as e:
            logger.error(f"Fout bij {bron_db}: {e}")

    conn_sdm.close()
    logger.info("Klaar met overzetten.")

overzetten_data()

2026-03-05 16:19:02.623 | INFO     | __main__:overzetten_data:23 - Start overzetten data naar SDM...
2026-03-05 16:19:02.638 | SUCCESS  | __main__:overzetten_data:35 - Overgezet: 20 rijen van BikeToDrive_1_Accessoireverkoop.db[Klant] naar SDM[Accessoire_Verkoop_Klant]
2026-03-05 16:19:02.651 | SUCCESS  | __main__:overzetten_data:35 - Overgezet: 100 rijen van BikeToDrive_1_Accessoireverkoop.db[Accessoire_Verkoop] naar SDM[Accessoire_Verkoop]
2026-03-05 16:19:02.662 | SUCCESS  | __main__:overzetten_data:35 - Overgezet: 25 rijen van BikeToDrive_2_Fietsverkoop.db[Klant] naar SDM[Fietsverkoop_Klant]
2026-03-05 16:19:02.664 | ERROR    | __main__:overzetten_data:38 - Fout bij BikeToDrive_2_Fietsverkoop.db: Execution failed
2026-03-05 16:19:02.666 | INFO     | __main__:overzetten_data:41 - Klaar met overzetten.


In [ ]:
import sqlite3
import pandas as pd
import os
from loguru import logger

def overzetten_alle_data():
    sdm_pad = 'SDM.db'
    logger.info("Start van de volledige data-pomp naar SDM...")

    # De volledige lijst met alle bronnen en doeltabellen
    mapping_config = {
        'BikeToDrive_1_Accessoireverkoop.db': {
            'Klant': 'Accessoire_Verkoop_Klant', 'Filiaal': 'Accessoire_Verkoop_Filiaal',
            'Monteur': 'Accessoire_Verkoop_Monteur', 'Leverancier': 'Accessoire_Verkoop_Leverancier',
            'Accessoire': 'Accessoire_Verkoop_Accessoire', 'Accessoire_Verkoop': 'Accessoire_Verkoop'
        },
        'BikeToDrive_2_Fietsverkoop.db': {
            'Klant': 'Fietsverkoop_Klant', 'Filiaal': 'Fietsverkoop_Filiaal',
            'Fiets': 'Fietsverkoop_Fiets', 'Fiets_Verkoop': 'Fiets_Verkoop'
        },
        'BikeToDrive_3_Onderhoud.db': {
            'Fabrikant': 'Onderhoud_Fabrikant', 'Fiets': 'Onderhoud_Fiets',
            'Filiaal': 'Onderhoud_Filiaal', 'Monteur': 'Onderhoud_Monteur', 'Onderhoud': 'Onderhoud'
        },
        'BikeToDrive_4_Accessoire_inkoop.db': {
            'Leverancier': 'Accessoire_Inkoop_Leverancier', 'Accessoire': 'Accessoire_Inkoop_Accessoire',
            'Accessoire_Inkoop': 'Accessoire_Inkoop'
        },
        'BikeToDrive_5_Fiets_Inkoop.db': {
            'Fabrikant': 'Fiets_Inkoop_Fabrikant', 'Fiets': 'Fiets_Inkoop_Fiets',
            'Fiets_Inkoop': 'Fiets_Inkoop'
        }
    }

    try:
        conn_sdm = sqlite3.connect(sdm_pad)
        
        for bron_db, tabellen in mapping_config.items():
            if not os.path.exists(bron_db):
                logger.error(f"Bestand niet gevonden: {bron_db}")
                continue
                
            conn_bron = sqlite3.connect(bron_db)
            logger.debug(f"Bezig met bronbestand: {bron_db}")
            
            for bron_tabel, sdm_tabel in tabellen.items():
                try:
                    df = pd.read_sql_query(f"SELECT * FROM {bron_tabel}", conn_bron)
                    # We gebruiken 'append' zodat de tabellen gevuld worden
                    df.to_sql(sdm_tabel, conn_sdm, if_exists='append', index=False)
                    logger.success(f"Gekopieerd: {len(df)} rijen naar {sdm_tabel}")
                except Exception as e:
                    logger.warning(f"Kon {bron_tabel} niet laden: {e}")
            
            conn_bron.close()
            
        conn_sdm.close()
        logger.success("Klaar! Alle beschikbare data is overgezet naar het SDM.")

    except Exception as e:
        logger.critical(f"Het proces is gestopt door een fout: {e}")

# DIT IS DE BELANGRIJKSTE REGEL: Hiermee start je de codessss

overzetten_alle_data()

2026-03-05 16:23:36.828 | INFO     | __main__:overzetten_alle_data:8 - Start van de volledige data-pomp naar SDM...
2026-03-05 16:23:36.829 | DEBUG    | __main__:overzetten_alle_data:44 - Bezig met bronbestand: BikeToDrive_1_Accessoireverkoop.db
2026-03-05 16:23:36.834 | WARNING  | __main__:overzetten_alle_data:53 - Kon Klant niet laden: Execution failed
2026-03-05 16:23:36.844 | SUCCESS  | __main__:overzetten_alle_data:51 - Gekopieerd: 4 rijen naar Accessoire_Verkoop_Filiaal
2026-03-05 16:23:36.851 | SUCCESS  | __main__:overzetten_alle_data:51 - Gekopieerd: 10 rijen naar Accessoire_Verkoop_Monteur
2026-03-05 16:23:36.866 | SUCCESS  | __main__:overzetten_alle_data:51 - Gekopieerd: 5 rijen naar Accessoire_Verkoop_Leverancier
2026-03-05 16:23:36.875 | SUCCESS  | __main__:overzetten_alle_data:51 - Gekopieerd: 10 rijen naar Accessoire_Verkoop_Accessoire
2026-03-05 16:23:36.877 | WARNING  | __main__:overzetten_alle_data:53 - Kon Accessoire_Verkoop niet laden: Execution failed
2026-03-05 16:

In [ ]:
import sqlite3
import pandas as pd
from loguru import logger

def overzetten_alle_data_fix():
    sdm_pad = 'SDM.db'
    
    mapping_config = {
        'BikeToDrive_1_Accessoireverkoop.db': {
            'Klant': 'Accessoire_Verkoop_Klant',
            'Accessoire_Verkoop': 'Accessoire_Verkoop'
        },
        'BikeToDrive_2_Fietsverkoop.db': {
            'Klant': 'Fietsverkoop_Klant',
            'Fiets': 'Fietsverkoop_Fiets',
            'Fiets_Verkoop': 'Fiets_Verkoop'
        }
       

    }

    conn_sdm = sqlite3.connect(sdm_pad)
    
    for bron_db, tabellen in mapping_config.items():
        conn_bron = sqlite3.connect(bron_db)
        for bron_tabel, sdm_tabel in tabellen.items():
            try:
                df = pd.read_sql_query(f"SELECT * FROM {bron_tabel}", conn_bron)
                # We gebruiken 'replace' om de kolom-fouten in je SDM te herstellen
                df.to_sql(sdm_tabel, conn_sdm, if_exists='replace', index=False)
                logger.success(f"FIXED & GEKOPIEERD: {len(df)} rijen naar {sdm_tabel}")
            except Exception as e:
                logger.error(f"Zelfs de fix werkte niet voor {bron_tabel}: {e}")
        conn_bron.close()
    
    conn_sdm.close()
    logger.info("De probleem-tabellen zijn nu hersteld en gevuld.")

overzetten_alle_data_fix()

2026-03-12 13:13:53.828 | SUCCESS  | __main__:overzetten_alle_data_fix:30 - FIXED & GEKOPIEERD: 20 rijen naar Accessoire_Verkoop_Klant
2026-03-12 13:13:53.845 | SUCCESS  | __main__:overzetten_alle_data_fix:30 - FIXED & GEKOPIEERD: 100 rijen naar Accessoire_Verkoop
2026-03-12 13:13:53.869 | SUCCESS  | __main__:overzetten_alle_data_fix:30 - FIXED & GEKOPIEERD: 25 rijen naar Fietsverkoop_Klant
2026-03-12 13:13:53.892 | SUCCESS  | __main__:overzetten_alle_data_fix:30 - FIXED & GEKOPIEERD: 75 rijen naar Fietsverkoop_Fiets
2026-03-12 13:13:53.915 | SUCCESS  | __main__:overzetten_alle_data_fix:30 - FIXED & GEKOPIEERD: 150 rijen naar Fiets_Verkoop
2026-03-12 13:13:53.916 | INFO     | __main__:overzetten_alle_data_fix:36 - De probleem-tabellen zijn nu hersteld en gevuld.
